# GLoVe, FastText and BPE

In [ ]:
!pip install datasets

In [72]:
import numpy as np
import pickle
import re
from collections import Counter
from pathlib import Path
from datasets import load_dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 282995.46 examples/s]


In [149]:
def clean_review(text):
    text = text.lower().replace("<br />", " ")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_reviews(texts, max_docs=None):
    docs = []
    for text in texts[:max_docs]:
        docs.append(clean_review(text).split())
    return docs


# Start small. Full IMDB is slow for this pure-Python implementation.
docs = tokenize_reviews(dataset["train"]["text"], max_docs=10000)
print(len(docs), "reviews")
print(docs[0][:30])

10000 reviews
['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', 'i', 'also', 'heard', 'that', 'at', 'first']


In [109]:
def build_vocab(docs, max_vocab=5000, min_count=5):
    counts = Counter(token for doc in docs for token in doc)
    words = [
        word for word, count in counts.most_common(max_vocab)
        if count >= min_count
    ]
    word_to_id = {word: i for i, word in enumerate(words)}
    id_to_word = {i: word for word, i in word_to_id.items()}
    return word_to_id, id_to_word


def build_cooccurrence(docs, window=5, max_vocab=5000, min_count=5):
    pair_counts = Counter()
    word_to_id, id_to_word = build_vocab(docs, max_vocab=max_vocab, min_count=min_count)

    for doc in docs:
        indexed = [word_to_id[t] for t in doc if t in word_to_id]
        for i, center in enumerate(indexed):
            for j in range(max(0, i - window), min(len(indexed), i + window + 1)):
                if i != j:
                    distance = abs(i - j)
                    pair_counts[(center, indexed[j])] += 1.0 / distance

    print("Vocab size:", len(word_to_id))
    print("Co-occurrence pairs:", len(pair_counts))
    return word_to_id, id_to_word, pair_counts



In [110]:

def glove_train(word_to_id, pair_counts, dim=50, epochs=25, lr=0.03, x_max=100, alpha=0.75, seed=0):
    n = len(word_to_id)
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(n, dim))
    W_tilde = rng.normal(0, 0.1, size=(n, dim))
    b = np.zeros(n)
    b_tilde = np.zeros(n)

    pairs = list(pair_counts.items())
    for epoch in range(epochs):
        rng.shuffle(pairs)
        total_loss = 0.0
        for (i, j), x_ij in pairs:
            weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0
            diff = W[i] @ W_tilde[j] + b[i] + b_tilde[j] - np.log(x_ij)
            coef = weight * diff
            total_loss += 0.5 * weight * diff * diff

            grad_W_i = coef * W_tilde[j]
            grad_W_tilde_j = coef * W[i]
            W[i] -= lr * grad_W_i
            W_tilde[j] -= lr * grad_W_tilde_j
            b[i] -= lr * coef
            b_tilde[j] -= lr * coef

        print(f"Epoch {epoch + 1}/{epochs} loss={total_loss / len(pairs):.4f}")

    return W + W_tilde

In [111]:
word_to_id, id_to_word, pair_counts = build_cooccurrence(
    docs,
    window=5,
    max_vocab=5000,
    min_count=5,
)

Vocab size: 5000
Co-occurrence pairs: 2816301


In [112]:
glove_vectors = glove_train(
    word_to_id,
    pair_counts,
    dim=50,
    epochs=10,
    lr=0.03,
    seed=42,
)

Epoch 1/10 loss=0.0325
Epoch 2/10 loss=0.0199
Epoch 3/10 loss=0.0168
Epoch 4/10 loss=0.0154
Epoch 5/10 loss=0.0144
Epoch 6/10 loss=0.0136
Epoch 7/10 loss=0.0129
Epoch 8/10 loss=0.0123
Epoch 9/10 loss=0.0117
Epoch 10/10 loss=0.0113


In [113]:
def nearest_words(word, word_to_id, id_to_word, vectors, topk=10):
    word = clean_review(word)
    if word not in word_to_id:
        raise ValueError(f"'{word}' is not in the vocabulary")

    target_idx = word_to_id[word]
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    vectors_norm = vectors / norms
    similarities = vectors_norm @ vectors_norm[target_idx]
    order = np.argsort(-similarities)

    results = []
    for idx in order:
        idx = int(idx)
        if idx == target_idx:
            continue
        results.append((id_to_word[idx], float(similarities[idx])))
        if len(results) == topk:
            break

    return results


nearest_words("discover", word_to_id, id_to_word, glove_vectors, topk=10)

[('needless', 0.6461168971548693),
 ('desperately', 0.6165491594708331),
 ('witness', 0.5640629041041457),
 ('attached', 0.5402884415421024),
 ('manage', 0.532922899276784),
 ('replace', 0.527547295917006),
 ('occur', 0.5208507198226683),
 ('randomly', 0.5153184873153801),
 ('warn', 0.5103457995032432),
 ('wanna', 0.5079349334117398)]

## FastText: Subwords of a Word

In [114]:

def char_ngrams(word, n_min=3, n_max=6):
    wrapped = f"<{word}>"
    grams = {wrapped}
    for n in range(n_min, n_max + 1):
        for i in range(len(wrapped) - n + 1):
            grams.add(wrapped[i:i + n])
    return grams


def build_ngram_table_from_word_vectors(word_to_id, vectors):
    """Simple FastText-style table: each n-gram gets the average of words containing it."""
    sums = {}
    counts = Counter()

    for word, idx in word_to_id.items():
        for gram in char_ngrams(word):
            if gram not in sums:
                sums[gram] = np.zeros(vectors.shape[1])
            sums[gram] += vectors[idx]
            counts[gram] += 1

    return {gram: sums[gram] / counts[gram] for gram in sums}


def fasttext_vector(word, ngram_table):
    grams = char_ngrams(word)
    vecs = [ngram_table[g] for g in grams if g in ngram_table]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)


def nearest_fasttext(word, vocabulary, ngram_table, topk=10):
    target = fasttext_vector(word, ngram_table)
    if target is None:
        raise ValueError(f"No known character n-grams for '{word}'")

    results = []
    target_norm = np.linalg.norm(target) + 1e-9
    for candidate in vocabulary:
        if candidate == word:
            continue
        vec = fasttext_vector(candidate, ngram_table)
        if vec is None:
            continue
        score = float((target @ vec) / (target_norm * (np.linalg.norm(vec) + 1e-9)))
        results.append((candidate, score))

    return sorted(results, key=lambda x: x[1], reverse=True)[:topk]



In [115]:
ngram_table = build_ngram_table_from_word_vectors(word_to_id, glove_vectors)
vocabulary = list(word_to_id.keys())

nearest_fasttext("movie", vocabulary, ngram_table, topk=10)

[('film', 0.927645170184542),
 ('this', 0.9232317104483955),
 ('movies', 0.8999149055637243),
 ('however', 0.8625640773937265),
 ('it', 0.8507224621028189),
 ('but', 0.8369401107664398),
 ('one', 0.812486155599408),
 ('making', 0.8076392361991482),
 ('most', 0.8039764703843574),
 ('that', 0.7988672204263335)]

In [ ]:
def save_glove_fasttext_bundle(
    path,
    word_to_id,
    id_to_word,
    glove_vectors,
    ngram_table,
    metadata=None,
):
    """Save trained GloVe vectors and the FastText-style n-gram table."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    bundle = {
        "word_to_id": word_to_id,
        "id_to_word": id_to_word,
        "glove_vectors": glove_vectors,
        "ngram_table": ngram_table,
        "metadata": metadata or {},
    }

    with path.open("wb") as f:
        pickle.dump(bundle, f)

    print(f"Saved model bundle to {path}")


def load_glove_fasttext_bundle(path):
    """Load a saved GloVe/FastText-style model bundle."""
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


model_path = "models/wt2_glove_fasttext.pkl"

save_glove_fasttext_bundle(
    model_path,
    word_to_id=word_to_id,
    id_to_word=id_to_word,
    glove_vectors=glove_vectors,
    ngram_table=ngram_table,
    metadata={
        "dataset": "Salesforce/wikitext wikitext-2-v1",
        "split": "train",
        "embedding_dim": int(glove_vectors.shape[1]),
        "vocab_size": len(word_to_id),
        "fasttext_note": "ngram_table is derived from trained GloVe word vectors, not separately trained with FastText skip-gram.",
    },
)

In [ ]:
# Later, in this notebook or another use case, load the saved model like this.
bundle = load_glove_fasttext_bundle("models/wt2_glove_fasttext.pkl")

loaded_word_to_id = bundle["word_to_id"]
loaded_id_to_word = bundle["id_to_word"]
loaded_glove_vectors = bundle["glove_vectors"]
loaded_ngram_table = bundle["ngram_table"]

print(bundle["metadata"])

print("GloVe nearest:")
print(nearest_words("the", loaded_word_to_id, loaded_id_to_word, loaded_glove_vectors, topk=10))

print("\nFastText-style nearest:")
print(nearest_fasttext("the", list(loaded_word_to_id.keys()), loaded_ngram_table, topk=10))

## BPE: Byte Pair Encoding 

is a data compression and tokenization algorithm that converts text into smaller, frequently occurring chunks called "subwords".


### How BPE Works
1. Instead of memorizing entire words or breaking text into individual letters, BPE builds a "subword vocabulary" using a repetitive statistical process: 

2. Character Breakdown: It begins by splitting every word into individual characters.
3. Frequency Counting: It scans a massive dataset to identify which pairs of adjacent characters (or bytes) appear together most frequently.
4. Merging: It merges the most frequent pair into a single new token and adds it to its vocabulary.
5. Iteration: It repeats this process—finding the most common pairs and merging them—until it reaches a predefined vocabulary size (e.g., 50,000 to 100,000 tokens). 

In [121]:
### This code is written by codex

def learn_bpe(corpus, k_merges):
    vocab = Counter()
    for word, freq in corpus.items():
        tokens = tuple(word) + ("</w>",)
        vocab[tokens] = freq

    merges = []
    for _ in range(k_merges):
        pair_freq = Counter()
        for tokens, freq in vocab.items():
            for a, b in zip(tokens, tokens[1:]):
                pair_freq[(a, b)] += freq
        if not pair_freq:
            break
        best = pair_freq.most_common(1)[0][0]
        merges.append(best)

        new_vocab = Counter()
        for tokens, freq in vocab.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == best:
                    new_tokens.append(tokens[i] + tokens[i + 1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_vocab[tuple(new_tokens)] = freq
        vocab = new_vocab
    return merges


def apply_bpe(word, merges):
    tokens = list(word) + ["</w>"]
    for a, b in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


def build_bpe_vocab(corpus, merges):
    """
    Builds subword vocabulary from a word-frequency corpus after applying BPE.
    """
    subwords = Counter()

    for word, freq in corpus.items():
        tokens = apply_bpe(word, merges)
        for token in tokens:
            subwords[token] += freq

    subword_to_id = {tok: i for i, tok in enumerate(subwords)}
    id_to_subword = {i: tok for tok, i in subword_to_id.items()}

    return subword_to_id, id_to_subword, subwords


def initialize_subword_vectors(subword_to_id, dim=50, seed=42):
    """
    Randomly initialize one vector per BPE token.
    """
    rng = np.random.default_rng(seed)

    vectors = rng.normal(
        loc=0.0,
        scale=0.1,
        size=(len(subword_to_id), dim)
    )

    return vectors


def word_to_bpe_vector(word, merges, subword_to_id, subword_vectors):
    """
    Converts a word into a vector by averaging its BPE subword vectors.
    """
    tokens = apply_bpe(word, merges)

    ids = [
        subword_to_id[tok]
        for tok in tokens
        if tok in subword_to_id
    ]

    if not ids:
        return None

    vec = subword_vectors[ids].mean(axis=0)

    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec

    return vec / norm


def build_word_vectors(corpus, merges, subword_to_id, subword_vectors):
    """
    Builds one vector per word in the corpus.
    """
    word_vectors = {}

    for word in corpus:
        vec = word_to_bpe_vector(
            word,
            merges,
            subword_to_id,
            subword_vectors
        )

        if vec is not None:
            word_vectors[word] = vec

    return word_vectors


def nearest_words_bpe(query_word, word_vectors, merges, subword_to_id, subword_vectors, top_k=10):
    """
    Finds nearest words to a query word using cosine similarity.
    """
    query_vec = word_to_bpe_vector(
        query_word,
        merges,
        subword_to_id,
        subword_vectors
    )

    if query_vec is None:
        print(f"No vector found for: {query_word}")
        return []

    scores = []

    for word, vec in word_vectors.items():
        if word == query_word:
            continue

        similarity = np.dot(query_vec, vec)
        scores.append((word, similarity))

    scores.sort(key=lambda x: x[1], reverse=True)

    return scores[:top_k]

In [125]:
bpe_corpus = Counter(token for doc in docs for token in doc)

merges = learn_bpe(bpe_corpus, k_merges=100)

print("First 20 learned merges:")
print(merges[:20])

print("\nTokenized examples:")
for word in ["movie", "movies", "unbelievable", "boring", "excellent"]:
    print(word, "->", apply_bpe(clean_review(word), merges))

First 20 learned merges:
[('e', '</w>'), ('s', '</w>'), ('t', 'h'), ('t', '</w>'), ('d', '</w>'), ('i', 'n'), ('y', '</w>'), ('e', 'r'), ('a', 'n'), ('th', 'e</w>'), ('o', '</w>'), ('o', 'n'), ('e', 'n'), ('o', 'r'), ('g', '</w>'), ('o', 'u'), ('i', 's</w>'), ('in', 'g</w>'), ('a', '</w>'), ('a', 'r')]

Tokenized examples:
movie -> ['movie</w>']
movies -> ['movi', 'es</w>']
unbelievable -> ['un', 'be', 'li', 'ev', 'ab', 'le</w>']
boring -> ['b', 'or', 'ing</w>']
excellent -> ['e', 'x', 'c', 'e', 'l', 'l', 'en', 't</w>']


In [126]:
# 1. Count words from your tokenized IMDB docs
bpe_corpus = Counter(token for doc in docs for token in doc)

# 2. Learn BPE merges
merges = learn_bpe(bpe_corpus, k_merges=100)

# 3. Build the BPE subword vocab
subword_to_id, id_to_subword, subword_counts = build_bpe_vocab(bpe_corpus, merges)

# 4. Initialize subword vectors
subword_vectors = initialize_subword_vectors(subword_to_id, dim=50, seed=42)

# 5. Build one vector per word
word_vectors = build_word_vectors(
    bpe_corpus,
    merges,
    subword_to_id,
    subword_vectors
)

In [130]:
word = "dance"
nearest_words_bpe(word, word_vectors, merges, subword_to_id, subword_vectors)

[('candace', np.float64(0.9146065852511246)),
 ('dane', np.float64(0.887820021163254)),
 ('candidate', np.float64(0.8758603450220893)),
 ('cane', np.float64(0.8737664459383778)),
 ('ance', np.float64(0.8737664459383778)),
 ('riddance', np.float64(0.8686892709921743)),
 ('dominance', np.float64(0.8437770922208304)),
 ('accordance', np.float64(0.823865979695283)),
 ('arcane', np.float64(0.822457863809033)),
 ('hindrance', np.float64(0.8215982765805597))]